# MvDEC REUTERS release-L2 normalization diagnostic - seed 42

Runs exactly one controlled GPU training experiment: `mvdec_2025_paper_resolved_release_l2_full_batch_v1` on REUTERS seed 42. It is derived from the paper-resolved full-batch protocol and changes only input normalization from paper mean-squared-norm-one to release-style row L2 normalization. Architecture, Equation 11 objective, raw latent averaging, K-Means schedule, assignment stopping rule, and full-batch refinement remain unchanged. This is a normalization diagnostic, not an exact-paper reproduction.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle GPU before running this training notebook."


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

TRAINING_COMMIT = "c6252ad95d3fe1395ceaee3e6f26e10043c23c0f"
PROTOCOL = "mvdec_2025_paper_resolved_release_l2_full_batch_v1"
REPO = Path("/kaggle/working/mvdec_fpmax")
OUTPUT = Path("/kaggle/working/mvdec-reuters-release-l2-seed42")
if REPO.exists():
    shutil.rmtree(REPO)
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
subprocess.run(
    ["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)],
    check=True,
)
subprocess.run(["git", "checkout", "--detach", TRAINING_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
assert resolved == TRAINING_COMMIT, (resolved, TRAINING_COMMIT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"],
    check=True,
)
subprocess.run(
    [sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"],
    cwd=REPO,
    check=True,
)


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_mvdec_contract.py",
        "tests/test_mvdec_preprocessing.py",
        "-k",
        "release_l2_diagnostic or paper_resolved_config_records_full_batch",
    ],
    cwd=REPO,
    check=True,
)


In [ ]:
command = [
    sys.executable,
    "-u",
    "src/representation_learning/MVDEC_dense.py",
    "REUTERS",
    "--runs",
    "1",
    "--seed",
    "42",
    "--protocol",
    PROTOCOL,
    "--progress-interval",
    "100",
    "--dataset-root",
    "external_repos/DEKM/datasets",
    "--public-output-dir",
    str(OUTPUT),
]
subprocess.run(command, cwd=REPO, check=True, timeout=4 * 60 * 60)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pipeline.mvdec_audit",
        "--root",
        str(OUTPUT),
        "--dataset-root",
        "external_repos/DEKM/datasets",
        "--expected-datasets",
        "REUTERS",
        "--expected-protocol-id",
        PROTOCOL,
        "--expected-training-commit",
        TRAINING_COMMIT,
        "--expected-seed",
        "42",
        "--require-all-complete",
    ],
    cwd=REPO,
    check=True,
)


In [ ]:
import hashlib
import json
import pickle

import numpy as np
import pandas as pd

run_dirs = sorted((OUTPUT / "reuters" / PROTOCOL).glob("*/seed_42"))
assert len(run_dirs) == 1, run_dirs
run_dir = run_dirs[0]
manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
assert manifest["status"] == "complete", manifest
assert manifest["training_commit"] == TRAINING_COMMIT
assert manifest["protocol_id"] == PROTOCOL
assert manifest["seed"] == 42
assert manifest["source_id"] == "dekm_google_drive_reuters_10k"
assert manifest["source_sha256"] == "427993db6b0544dfce1b8e3985e63e248d4d006de122cb202c504489d951706b"
contract = manifest["config"]["protocol_contract"]
assert contract["preprocessing"]["input_normalization"] == "release_l2_unit_norm"
assert contract["schedule"]["refinement"]["batch_size"] == "full_dataset"
assert contract["schedule"]["refinement"]["batch_size_policy"] == "full_dataset"
assert contract["architecture"].get("fusion", "mvdec2025_encoder_average") == "mvdec2025_encoder_average"

artifact_path = run_dir / manifest["paths"]["artifact"]
with artifact_path.open("rb") as handle:
    artifact = pickle.load(handle)
assert artifact["protocol_contract"]["preprocessing"]["input_normalization"] == "release_l2_unit_norm"
assert artifact["fusion_contract"] == "mvdec2025_encoder_average"

metrics = manifest["metrics"]
comparison = pd.DataFrame(
    [
        {"reference": "release-L2 diagnostic seed 42", "acc": metrics["fused_acc"], "nmi": metrics["fused_nmi"]},
        {"reference": "paper-resolved RMS seed 42", "acc": 0.7194, "nmi": 0.5286},
        {"reference": "older release-L2 A2 seed 42", "acc": 0.7621, "nmi": 0.6002},
        {"reference": "paper fused result", "acc": 0.7813, "nmi": 0.6141},
    ]
)
comparison.to_csv(OUTPUT / "normalization_comparison.csv", index=False)
summary = {
    "training_commit": TRAINING_COMMIT,
    "protocol_id": PROTOCOL,
    "seed": 42,
    "config_hash": manifest["config_hash"],
    "source_id": manifest["source_id"],
    "source_sha256": manifest["source_sha256"],
    "runtime_seconds": manifest["runtime_seconds"],
    **metrics,
}
(OUTPUT / "normalization_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
display(comparison)
print(json.dumps(summary, indent=2))


In [ ]:
zip_base = "/kaggle/working/mvdec-reuters-release-l2-seed42"
zip_path = Path(shutil.make_archive(zip_base, "zip", OUTPUT.parent, OUTPUT.name))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = Path(str(zip_path) + ".sha256")
sha_path.write_text(f"{digest}  {zip_path.name}\n", encoding="ascii")
print("ZIP:", zip_path)
print("SHA-256:", digest)
